## प्रस्तावना 

हा धडा असे समजावून सांगेल: 
- फंक्शन कॉल म्हणजे काय आणि त्याचे वापरप्रकरणे काय आहेत 
- Azure OpenAI वापरून फंक्शन कॉल कसा तयार करायचा 
- फंक्शन कॉलला अॅप्लिकेशनमध्ये कसे एकत्रित करायचे 

## शिकण्याचे उद्दिष्टे 

हा धडा पूर्ण केल्यानंतर तुम्हाला हे कसे करायचे येईल आणि समजलेले असेल: 

- फंक्शन कॉलचा वापर करण्याचा हेतू 
- Azure Open AI सेवा वापरून फंक्शन कॉल सेटअप करणे 
- तुमच्या अॅप्लिकेशनच्या वापरप्रकरणासाठी प्रभावी फंक्शन कॉल डिझाइन करणे 


## फंक्शन कॉल्स समजून घेणे

या धड्याबाबत, आम्हाला आमच्या शिक्षण स्टार्टअपने एक वैशिष्ट्य तयार करायचे आहे जे वापरकर्त्यांना तांत्रिक कोर्सेस शोधण्यासाठी चॅटबॉट वापरण्याची परवानगी देते. आम्ही त्यांच्या कौशल्याच्या पातळी, सध्याच्या भूमिके आणि आवडत्या तंत्रज्ञानानुसार कोर्सेसची शिफारस करू.

हे पूर्ण करण्यासाठी आम्ही खालील संयोजन वापरणार आहोत:
 - `Azure Open AI` वापरकर्त्यांसाठी चॅट अनुभव तयार करण्यासाठी
 - `Microsoft Learn Catalog API` वापरकर्त्यांच्या विनंतीनुसार कोर्सेस शोधण्यासाठी मदत करण्यासाठी
 - `Function Calling` वापरकर्त्याच्या क्वेरीला घेऊन API विनंती करण्यासाठी फंक्शनमध्ये पाठविण्यासाठी.

सुरू करण्यासाठी, पाहूया आपण सुरुवातीला फंक्शन कॉलिंग का वापरू इच्छितो:


### फंक्शन कॉलिंग का आहे 

जर तुम्ही या कोर्समधील इतर कोणतेही धडा पूर्ण केला असेल, तर तुम्हाला मोठ्या भाषा मॉडेल्स (LLMs) वापरण्याची क्षमता कदाचित कळलेली असेल. आशा आहे की तुम्हाला त्यांच्या काही मर्यादाही समजतात. 

फंक्शन कॉलिंग हे Azure Open AI Service चे एक वैशिष्ट्य आहे जे खालील मर्यादा दूर करण्यासाठी आहे: 
1) सुसंगत प्रतिसाद स्वरूप 
2) चॅट संदर्भात अनुप्रयोगाच्या इतर स्रोतांमधील डेटाचा वापर करण्याची क्षमता 

फंक्शन कॉलिंगपूर्वी, LLM कडून प्रतिसाद असंरचित आणि असुसंगत होता. विकासकांना प्रत्येक प्रतिसादाच्या बदलासाठी जास्त क्लिष्ट सत्यापन कोड लिहावा लागायचा. 

वापरकर्ते "स्टॉकहोममधील सध्याचे हवामान काय आहे?" असे प्रश्न विचारू शकत नव्हते. कारण मॉडेल्सना केवळ त्या वेळेपर्यंतचे प्रशिक्षित डेटा उपलब्ध होते. 

खालील उदाहरण पाहूया जे या समस्येचे स्पष्टीकरण करते: 

समजा आपण विद्यार्थ्यांचा डेटाबेस तयार करू इच्छितो जेणेकरून त्यांना योग्य अभ्यासक्रम सुचवता येईल. खाली आपण दोन विद्यार्थ्यांच्या वर्णनाला पाहू शकतो जे आपल्या मध्ये दिलेल्या डेटामध्ये खूपच सारखे आहेत.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


आम्हाला हे डेटा पार्स करण्यासाठी LLM कडे पाठवायचे आहे. नंतर हे आमच्या अनुप्रयोगात API ला पाठवण्यासाठी किंवा डेटाबेसमध्ये साठवण्यासाठी वापरले जाऊ शकते.

आपण दोन सारखे प्रॉम्प्ट तयार करू ज्यात आम्ही LLM ला आम्हाला कोणती माहिती पाहिजे ते सांगू:


आम्हाला हे आमच्या उत्पादनासाठी महत्त्वाच्या भागांचे विश्लेषण करण्यासाठी LLM कडे पाठवायचे आहे. म्हणून आम्ही LLM ला सूचना देण्यासाठी दोन अगदी समान प्रॉम्प्ट तयार करू शकतो: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


या दोन प्रॉम्प्ट तयार केल्यानंतर, आपण त्यांना `client.responses.create` वापरून LLM कडे पाठवू. आम्ही प्रॉम्प्ट `input` व्हेरिएबलमध्ये संग्रहित करतो आणि भूमिका `user` म्हणून नियुक्त करतो. हे वापरकर्त्याने चॅटबॉटला लिहिलेल्या संदेशाचा अनुकरण करण्यासाठी आहे.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

आता आपण दोन्ही विनंत्या LLM ला पाठवू शकतो आणि आम्हाला प्राप्त झालेल्या प्रतिसादाचे परीक्षण करू शकतो.


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



जरी प्रॉम्प्ट्स तीच असली आणि वर्णने सारखी असली तरी, आपण `Grades` गुणधर्माचे विविध स्वरूप प्राप्त करू शकतो.

जर आपण वरची सेल अनेक वेळा चालवली, तर स्वरूप `3.7` किंवा `3.7 GPA` असे असू शकते.

हे कारण आहे की LLM लिहिलेल्या प्रॉम्प्टच्या स्वरूपात असंरचित डेटा घेतो आणि परतही असंरचित डेटा देतो. आपल्याला संरचित स्वरूप हवे जेणेकरून आपण या डेटा साठवताना किंवा वापरताना काय अपेक्षित आहे हे समजू शकेल.

फंक्शन कॉलिंग वापरून, आपण संरचित डेटा परत मिळतो याची खात्री करू शकतो. फंक्शन कॉलिंग वापरताना, LLM प्रत्यक्षात कोणतेही फंक्शन्स कॉल किंवा चालवत नाही. त्याऐवजी, आपण LLM च्या प्रतिसादांसाठी एक संरचना तयार करतो. नंतर आपण त्या संरचित प्रतिसादांचा वापर करून आमच्या अनुप्रयोगांमध्ये कोणता फंक्शन चालवायचा ते ठरवतो.
 


![कार्यप्रवाह कॉल फ्लो आकृती](../../../../translated_images/mr/Function-Flow.083875364af4f4bb.webp)


नंतर आपण कार्यातून परत आलेले काहीही घेऊ शकतो आणि ते परत LLM कडे पाठवू शकतो. LLM नंतर वापरकर्त्याच्या प्रश्नाचे उत्तर देण्यासाठी नैसर्गिक भाषेचा वापर करून प्रतिसाद देईल.


### फंक्शन कॉल वापरण्याचे वापरप्रकरणे 

**बाह्य साधने कॉल करणे** 
चॅटबॉट्स वापरकर्त्यांच्या प्रश्नांची उत्तरे देण्यात उत्कृष्ट आहेत. फंक्शन कॉलिंग वापरून, चॅटबॉट्स वापरकर्त्यांकडून संदेश घेऊन विशिष्ट कार्य करू शकतात. उदाहरणार्थ, विद्यार्थी चॅटबॉटला "माझ्या शिक्षकाला ईमेल पाठवा की मला या विषयात अधिक मदतीची गरज आहे" असे विचारू शकतो. यासाठी `send_email(to: string, body: string)` नावाचे फंक्शन कॉल होऊ शकते


**API किंवा डेटाबेस क्वेरीज तयार करणे** 
वापरकर्ते नैसर्गिक भाषेचा वापर करून माहिती शोधू शकतात जी नंतर फॉर्मॅट केलेल्या क्वेरी किंवा API विनंतीत रूपांतरित केली जाते. उदाहरणार्थ, शिक्षक असे विचारू शकतो "शेवटची असाइनमेंट पूर्ण केलेले विद्यार्थी कोण आहेत?" जेव्हा `get_completed(student_name: string, assignment: int, current_status: string)` नावाचे फंक्शन कॉल होऊ शकते


**रचनाबद्ध डेटा तयार करणे** 
वापरकर्ते मजकूराचा किंवा CSV ब्लॉकचा वापर करून LLM कडून महत्त्वाची माहिती काढू शकतात. उदाहरणार्थ, विद्यार्थी शांतता करारांबद्दल व्हिकिपीडिया लेख रूपांतरित करून AI फ्लॅश कार्ड तयार करू शकतो. यासाठी `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नावाचे फंक्शन वापरले जाऊ शकते


## 2. तुमचा पहिला फंक्शन कॉल तयार करणे 

फंक्शन कॉल तयार करण्याच्या प्रक्रियेमध्ये 3 मुख्य टप्पे असतात: 
1. तुमच्या फंक्शन्सची यादी आणि वापरकर्त्याचा संदेश वापरून Chat Completions API कॉल करणे 
2. कृती करण्यासाठी म्हणजे फंक्शन किंवा API कॉल चालवण्यासाठी मॉडेलला आलेला प्रतिसाद वाचणे 
3. वापरकर्त्यास प्रतिसाद तयार करण्यासाठी तुमच्या फंक्शनकडून आलेल्या प्रतिसादासह Chat Completions API ला दुसरा कॉल करणे. 


![फंक्शन कॉलचा प्रवाह](../../../../translated_images/mr/LLM-Flow.3285ed8caf4796d7.webp)


### फंक्शन कॉलची घटक  

#### वापरकर्त्याचे इनपुट  

पहिला टप्पा म्हणजे वापरकर्त्याचा संदेश तयार करणे. हा टेक्स्ट इनपुटची किंमत घेऊन डायनॅमिकली दिला जाऊ शकतो किंवा तुम्ही येथे एक किंमत नियुक्त करू शकता. जर तुम्ही Chat Completions API सह प्रथमच काम करत असाल, तर आपल्याला संदेशाचा `role` आणि `content` परिभाषित करणे गरजेचे आहे.  

`role` हा `system` (नियम तयार करणे), `assistant` (मॉडेल) किंवा `user` (अंतिम वापरकर्ता) यापैकी कोणताही असू शकतो. फंक्शन कॉलिंगसाठी, आपण याला `user` म्हणून नियुक्त करू आणि एक उदाहरण प्रश्न देऊ.  


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फंक्शन्स तयार करणे.

पुढे आपण एक फंक्शन आणि त्या फंक्शनचे पॅरामीटर्स व्याख्यित करू. आपण येथे फक्त एक फंक्शन वापरणार आहोत ज्याचे नाव `search_courses` आहे परंतु आपण अनेक फंक्शन्स तयार करू शकता.

**महत्वाचे** : फंक्शन्स LLM कडे सिस्टम मेसेजमध्ये समाविष्ट केले जातात आणि ते तुम्हाला उपलब्ध असलेल्या टोकनांच्या प्रमाणात समाविष्ट असतील.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**व्याख्या**

`name` - कॉल करायची असलेली फंक्शनचे नाव.

`description` - फंक्शन कसे कार्य करते याचे वर्णन. येथे विशिष्ट आणि स्पष्ट असणे महत्त्वाचे आहे.

`parameters` - जे मूल्य आणि स्वरूप तुम्हाला मॉडेलने प्रतिसादादरम्यान तयार करायचे आहे त्यांची यादी.


`type` - त्या गुणधर्मांची डेटा प्रकार ज्या स्वरूपात साठवले जातील.

`properties` - विशिष्ट मूल्यांची यादी ज्याचा मॉडेल त्याच्या प्रतिसादासाठी वापर करेल.


`name` - गुणधर्माचे नाव जे मॉडेल त्याच्या स्वरूपित प्रतिसादात वापरेल.

`type` - या गुणधर्माचा डेटा प्रकार.

`description` - विशिष्ट गुणधर्माचे वर्णन.


**पर्यायी**

`required` - फंक्शन कॉल पूर्ण करण्यासाठी आवश्यक गुणधर्म.


### फंक्शन कॉल करणे  
फंक्शन परिभाषित केल्यानंतर, आता आपल्याला ते Chat Completion API कॉलमध्ये समाविष्ट करावे लागेल. हे आपण विनंतीमध्ये `functions` जोडून करतो. या बाबतीत `functions=functions`.  

`function_call` ला `auto` सेट करण्याचा पर्याय देखील आहे. याचा अर्थ आहे की आपण स्वतः ते ठरवण्याऐवजी वापरकर्त्याच्या संदेशावर आधारित कोणते फंक्शन कॉल करायचे ते LLM ला ठरवू देऊ.  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


आता आपण प्रतिसाद पाहूया आणि ते कसे स्वरूपित आहे ते पाहूया: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

तुम्ही पाहू शकता की फंक्शनचे नाव कॉल केले गेले आहे आणि वापरकर्त्याच्या संदेशातून, LLM ला फंक्शनच्या युक्तिवादाला बसणारी माहिती सापडली आहे. 


## 3.अॅप्लिकेशनमध्ये फंक्शन कॉल्स समक्रमित करणे. 


जेव्हा आपण LLM कडून स्वरूपित प्रतिसाद तपासला आहे, तेव्हा आता आपण याला अॅप्लिकेशनमध्ये समक्रमित करू शकतो. 

### प्रवाह व्यवस्थापन 

याला आपल्या अॅप्लिकेशनमध्ये समक्रमित करण्यासाठी, चला खालील पावले उचलूया: 

प्रथम, Open AI सेवांकडे कॉल करूयात आणि संदेश `response_message` नावाच्या चलामध्ये साठवूयात. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


आता आपण तो फंक्शन परिभाषित करणार आहोत जो Microsoft Learn API ला कॉल करेल आणि कोर्सची यादी प्राप्त करेल: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



एक सर्वोत्तम सराव म्हणून, नंतर आपण पाहू की मॉडेलला एखाद्या फंक्शनला कॉल करायचे आहे का. त्यानंतर, आपण उपलब्ध फंक्शन्सपैकी एक तयार करू आणि कॉल केलेल्या फंक्शनशी जुळवू. 
त्यानंतर आपण फंक्शनचे आर्ग्युमेंट्स घेऊन त्यांना LLM मधील आर्ग्युमेंट्सशी जुळवू.

शेवटी, आपण फंक्शन कॉल संदेश आणि `search_courses` संदेशाने परत आलेल्या मूल्यांना जोडू. यामुळे LLM ला सर्व माहिती मिळते जी त्याला
वापरकर्त्याला नैसर्गिक भाषेचा वापर करून प्रतिसाद देण्यासाठी आवश्यक आहे. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


आता आपण अद्यतनित संदेश LLM ला पाठवू ज्यामुळे आपल्याला API JSON स्वरूपित प्रतिसादाऐवजी नैसर्गिक भाषेतील प्रतिसाद प्राप्त होईल.


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड आव्हान

शानदार काम! Azure Open AI फंक्शन कॉलिंगचे तुमचे शिक्षण सुरू ठेवण्यासाठी तुम्ही पुढील गोष्टी तयार करू शकता: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst
 - फंक्शनचे आणखी काही पॅरामिटर्स जे अभ्यासकांना अधिक कोर्सेस शोधण्यात मदत करू शकतात. तुम्ही उपलब्ध API पॅरामिटर्स येथे पाहू शकता:
 - आणखी एक फंक्शन कॉल तयार करा जे अभ्यासकांकडून त्यांची स्थानिक भाषा सारखी अधिक माहिती घेतो
 - जेव्हा फंक्शन कॉल आणि/किंवा API कॉल योग्य कोर्सेस परत करत नाही तेव्हा त्रुटी हाताळणी तयार करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
